# 🚀 Korpusverarbeitung – Annotation mit spaCy

```{admonition} Hinweise zur Ausführung des Notebooks
:class: hinweis
Dieses Notebook kann auf unterschiedlichen Levels erarbeitet werden (siehe Abschnitt ["Technische Voraussetzungen"](../introduction/introduction_requirements)):

1. **Book-Only Mode:** Sie lesen das Notebook hier im "Jupyter Book", ohne den Code selbst auszuführen.
2. **Cloud Mode:** Klicken Sie **oben rechts in der Menüleiste** auf das Raketen-Symbol <span class="launch-colab-inline">🚀</span> und wählen Sie "Colab", um das Notebook auszuführen.
3. **Local Mode:** Klicken Sie **oben rechts in der Menüleiste** auf das Download-Symbol <span class="launch-ipynb-inline">↓</span> und wählen Sie ".ipynb", um das Notebook lokal auszuführen.
```

## Übersicht
Im Folgenden wird exemplarisch ein Text (txt-Datei) mit der Bibliothek [spaCy](https://spacy.io) annotiert. Dafür werden folgende Schritte durchgeführt:
1. Einlesen des Texts
2. Worthäufigkeiten ohne echte Tokenisierung
   * Aufteilen des Texts in Wörter auf Grundlage von Leerzeichen
   * Abfrage von Häufigkeiten
4. Annotation mit spaCy
   * Laden des Sprachmodells
   * Analysekomponenten auswählen
   * Text annotieren
   * Worthäufigkeiten anzeigen
5. Annotation speichern
6. Prozess für das gesamte Korpus ausführen

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
  
<b>Voraussetzungen zur Ausführung des Jupyter Notebooks</b>
<ol>
<li> Installieren der Bibliotheken </li>
<li>Laden der Daten (s.u.)</li>
<li>Pfad zu den Daten setzen</li>
</ol>
Zum Testen: Ausführen der Zelle „load libraries“ und der Sektion „Einlesen des Texts“. </br>
Alle Zellen, die mit 🚀 gekennzeichnet sind, werden nur bei der Ausführung des Notebooks in Colab / JupyterHub bzw. lokal ausgeführt. 
</details>

In [ ]:
#  🚀 Install libraries 
! pip install tqdm pandas numpy spacy bokeh requests

#  🚀 Load german language model for annotation
#! python -m spacy download de_core_news_sm

In [ ]:
# load libraries 
import json
import typing
import requests
from pathlib import Path
from time import time
from collections import OrderedDict, Counter

from tqdm import tqdm
import pandas as pd
import numpy as np
import spacy

from bokeh.io import output_notebook, show
from bokeh.layouts import column
from bokeh.models import CustomJS, TextInput, Div
from IPython.display import display, HTML

Bevor wir Daten herunterladen, definieren wir eine kleine Hilfsfunktion `download_file`. Sie lädt eine Datei plattformunabhängig – also auch unter Windows – aus dem Internet in einen Zielordner herunter und ersetzt damit das Kommando `wget`, das nicht auf allen Systemen (z.B. Windows) nativ verfügbar ist.

In [ ]:
# helper: download a single file (cross-platform replacement for `! wget -P`)
def download_file(url, target_dir):
    """Download the file at `url` into `target_dir`, keeping its original name."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / url.split("/")[-1]
    response = requests.get(url)
    response.raise_for_status()
    target_path.write_bytes(response.content)
    return target_path

## Einlesen des Textes
Um eine Datei mit Python bearbeiten zu können, muss die Datei zuerst ausgewählt werden, d.h. der [Pfad](https://de.wikipedia.org/wiki/Pfadname) zur Datei wird gesetzt und dann eingelesen werden. 

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Zuerst wird der Ordner angelegt, in dem die Textdateien gespeichert werden. Der Einfachheit halber wird die gleiche Datenablagestruktur wie in dem <a href="https://github.com/quadriga-dk/Text-Fallstudie-1/tree/main">GitHub Repository</a>, in dem die Daten gespeichert sind, vorausgesetzt. </br>
Der Text wird aus GitHub heruntergeladen und in dem Ordner <i>../data/txt/</i> abgespeichert. </br>
Der Pfad kann in der Variable <i>text_path</i> angepasst werden. Die einzulesenden Daten müssen die Endung `.txt` haben. </br>
</details>

In [ ]:
# 🚀 Create data directory path
corpus_dir = Path("../data/txt")
corpus_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 🚀 Load the txt file from GitHub 
download_file("https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/data/txt/SNP2719372X-19181015-0-0-0-0.txt", "../data/txt")

In [ ]:
# set the path to file to be processed
text_path = Path("../data/txt/SNP2719372X-19181015-0-0-0-0.txt")

In [ ]:
# read text and print some parts of the text
if text_path.is_file():
    text = text_path.read_text(encoding="utf-8")
    print(f"Textauszug:\n {text[10280:10400]}")
else:
    print("The file path does not exist. Set the variable text_path to an existing path.")

## Worthäufigkeiten ohne echte Tokenisierung

### Text in Wörter aufteilen
Der einfachste Weg, einen Text automatisch in Wörter aufzuteilen, ist anzunehmen, dass Wörter durch Leerzeichen getrennt sind.

In [ ]:
# split the text into words by space
words = text.split()

**Prüfen**: Wie sieht die Wortliste aus?

In [ ]:
# print the 100th up the 120th words
words[100:120]

Wie viele Wörter gibt es insgesamt?

In [ ]:
# print the length of the word list
len(words)

Wie zu sehen ist, hat diese Art der „falschen“ Tokenisierung den Nachteil, dass Satzzeichen nicht von Wörtern abgetrennt werden. \
Die Wortanzahl ist dementsprechend auch nicht genau. 

### 🚀 Selbst ausprobieren

Now that you understood how Naive Split and spaCy work, try out the interactive exercise below. In this interactive element, the text box already contains some default texts from the same document. The two buttons will allow you to see how the split occurs in case of a Native Split and when you use spaCy. The accompanying graph also visualises the change in token frequency which happens due to the use of spaCy. Feel free to try with different text combinations.

In [ ]:
from IPython.display import display, HTML

display(HTML(r"""
<div id="token-demo-v2" style="font-family: system-ui, monospace; max-width:1000px; margin: 8px 0;">
  <p style="font-size: 1.5rem; margin: 0 0 6px 0;">Tokenization + Frequency</p>

  <textarea id="inputTextV2" style="width:100%; height:90px; margin-bottom:10px;">
Die deutscchen Nach- uten, von einzelnen am Feinde gelassenen Tutterien und Geschüßen unterstüßt, hielten die rsichtig und langsam nachdrängenden in respektooller Entfernung, Die beschränkten sich in der Hauptsache de Ortschaften im deutschen Hinter- <-> mit Bombengeschwadern anzugreisen. der Zwischenzeit wurde von den Deutschen x Ruhe die ungeheure Arbeit der Rücdverlegung der Linien durchgeführt</textarea>

  <div style="margin-bottom:10px;">
    <button class="tokBtn" onclick="showNaiveV2()">Naive Split</button>
    <button class="tokBtn" onclick="showSpacyV2()">spaCy-like</button>
    <button class="tokBtn" onclick="clearV2()">Clear</button>
    <button class="tokBtn" onclick="restoreV2()">Restore Original Text</button>
  </div>
  <div id="spacyInfoV2"
     style="margin-top:8px; margin-bottom:5px; font-size:1.0rem;">
  </div>

  <div id="outputV2"
       style="white-space:pre-wrap; padding:10px; border-radius:6px;
              background:#1f1f1f; color:#eee; border:1px solid #2f2f2f;"></div>

  <div id="countV2" style="margin-top:10px;"></div>

  <div id="chartV2" style="margin-top:15px;"></div>

</div>

<style>
#token-demo-v2 .tokBtn {
  background:#2d3b4f; color:#fff; border:none;
  padding:6px 10px; margin-right:6px;
  border-radius:5px; cursor:pointer; font-size:0.9rem;
}

#token-demo-v2 .tok {
  padding:2px 4px; margin:2px;
  border-radius:3px; display:inline-block;
}

#token-demo-v2 .naive { background:#ff6b6b; color:#000; }
#token-demo-v2 .spacy { background:#7effa2; color:#000; }
</style>

<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>

<script>
(function(){

  const defaultText = `Die deutscchen Nach- uten, von einzelnen am Feinde gelassenen Tutterien und Geschüßen unterstüßt, hielten die rsichtig und langsam nachdrängenden in respektooller Entfernung, Die beschränkten sich in der Hauptsache de Ortschaften im deutschen Hinter- <-> mit Bombengeschwadern anzugreisen. der Zwischenzeit wurde von den Deutschen x Ruhe die ungeheure Arbeit der Rücdverlegung der Linien durchgeführt`;  

  function render(tokens, type, changedTokens = new Set()){
      const box = document.getElementById("outputV2");
    
      box.innerHTML = tokens.map(t => {
    
        let cls = type;
    
        if(type === "spacy" && changedTokens.has(t)){
          cls = "naive";
        }
    
        return `<span class="tok ${cls}">${t}</span>`;
    
      }).join(" ");
    
      const label = type === "naive" ? "Word count:" : "Token count:";
    
      document.getElementById("countV2").innerHTML =
        `<strong>${label}</strong> ${tokens.length}`;
  }

  

  function renderChart(freqs, title){
    const x = Object.keys(freqs);
    const y = Object.values(freqs);

    const data = [{
      x: x,
      y: y,
      type: "bar"
    }];

    const layout = {
      title: title,
      margin: { t: 40 }
    };

    Plotly.newPlot("chartV2", data, layout);
  }

  window.showNaiveV2 = function(){
    document.getElementById("spacyInfoV2").innerHTML = "";
    const text = document.getElementById("inputTextV2").value;
    const tokens = text.split(/\s+/).filter(t => t.length > 0);

    render(tokens, "naive");

    const freq = {};
    tokens.forEach(t => freq[t] = (freq[t] || 0) + 1);

    renderChart(freq, "Naive Frequencies");
  };

  window.showSpacyV2 = function(){
      const text = document.getElementById("inputTextV2").value;
    
      const naiveTokens = text
        .split(/\s+/)
        .filter(t => t.length > 0);
    
      const tokens = text
        .replace(/([.,!?;:"()<>-])/g, " $1 ")
        .split(/\s+/)
        .filter(t => t.length > 0);
    
      const changedTokens = new Set();
    
      naiveTokens.forEach(nt => {
    
        const splitVersion = nt
          .replace(/([.,!?;:"()<>-])/g, " $1 ")
          .split(/\s+/)
          .filter(t => t.length > 0);
    
        if(splitVersion.length > 1){
          splitVersion.forEach(t => changedTokens.add(t));
        }
    
      });

      document.getElementById("spacyInfoV2").innerHTML =
      "<em>Red tokens indicate where spaCy performed additional splitting compared to the Naive Split, while green tokens indicate tokens that remained unchanged.</em>";
    
      render(tokens, "spacy", changedTokens);
    
      const freq = {};
      tokens.forEach(t => freq[t] = (freq[t] || 0) + 1);
    
      renderChart(freq, "Tokenized Frequencies");
  };

  window.clearV2 = function(){
    document.getElementById("spacyInfoV2").innerHTML = "";
    document.getElementById("outputV2").innerHTML = "";
    document.getElementById("countV2").innerText = "";
    document.getElementById("chartV2").innerHTML = "";
    document.getElementById("inputTextV2").value = "";
  };

  window.restoreV2 = function(){

    document.getElementById("inputTextV2").value = defaultText;
    document.getElementById("spacyInfoV2").innerHTML = "";
    document.getElementById("outputV2").innerHTML = "";
    document.getElementById("countV2").innerText = "";
    document.getElementById("chartV2").innerHTML = "";
  };

})();
</script>
"""))

### Anzeigen von Worthäufigkeiten
Auf Grundlage dieser Wortliste kann trotzdem schon eine erste basale Häufigkeitenabfrage erfolgen. Dafür werden die Wörter zuerst gezählt. 

In [ ]:
# Count the words with Counter and save the result to a variable
word_frequencies = Counter(words)

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Um die Häufigkeit nur mit Python abzufragen, kann folgende Zeile ausgeführt werden:
</details>

In [ ]:
# 🚀 get the number of the word "Grippe" in the word frequencies 
word_frequencies["Grippe"]

Dann kann die Häufigkeit abgefragt werden:

In [ ]:
# Ensure Bokeh output is displayed in the notebook
output_notebook()

# Convert the dictionary to a JSON string to be passed to javascript
word_freq_json = json.dumps(word_frequencies)

# Create the text input widget
text_input = TextInput(value='', title="Geben Sie ein Wort ein:")

# Create a Div to display the frequency
frequency_display = Div(text="Häufigkeit: ")

# JavaScript callback to update the frequency display
# Only needed for graphical interface 
callback = CustomJS(args=dict(frequency_display=frequency_display, text_input=text_input), code=f"""
    var word = text_input.value.trim();

    // Parse the word frequency dictionary from Python
    var word_freq = {word_freq_json};

    var frequency = word in word_freq ? word_freq[word] : "Nicht gefunden";
    frequency_display.text = "Häufigkeit: " + frequency;
""")

text_input.js_on_change('value', callback)

# Layout and display
layout = column(text_input, frequency_display)
show(layout)

## Annotation mit spaCy
Um eine präzisere Einteilung in Wörter zu erhalten (Tokenisierung) und um flektierte Wörter aufeinander abbildbar zu machen (Lemmatisierung), wird der Text im Folgenden durch die Bibliothek [spaCy](https://spacy.io/) annotiert. Dafür werden folgende Schritte ausgeführt:
1. Das sprachspezifische Modell wird geladen. Wir arbeiten mit dem weniger akkuraten, aber schnellsten spaCy Modell `de_core_news_sm`. 
2. Für eine erhöhte Annotationsgeschwindigkeit werden nur bestimmte Analysekomponenten geladen. Dies ist vor allem für größere Textmengen sinnvoll.
3. Der Text wird annotiert und die Token sowie die dazugehörigen Lemmata werden extrahiert.

### Sprachmodell laden
Das sprachspezifische Modell wird geladen. Es handelt sich dabei um das am wenigsten akkurate aber schnellste Modell. 

In [ ]:
nlp = spacy.load('de_core_news_sm')

### Analysekomponenten auswählen
Es werden einige Analysekomponenten wie z. B. das Aufteilen des Texts in Sätze (sentencizer) oder die [Named Entity Recognition](https://en.wikipedia.org/wiki/Named-entity_recognition) (ner) ausgeschlossen, da diese für die Tokenisierung und die Lemmatisierung nicht benötigt werden. Der Ausschluss der Komponenten erhöht die Annotationsgeschwindigkeit. 

In [ ]:
disable_components = ['ner', 'morphologizer', 'attribute_ruler', 'sentencizer']

### Annotieren des Textes: Token, Lemma
Der ausgewählte Text wird mit spaCy annotiert und die Token sowie die dazugehörigen Lemmata werden extrahiert und in einer Tabelle gespeichert. Das Tabellenformat wurde gewählt, da sich darin gut relationale Daten speichern lassen.

In [ ]:
# get the current time to display how long the annotation took
current = time()

# annotate with spacy
doc = nlp(text)

# extract tokens and lemmata, save them to a dictionary
text_annotated = {}
text_annotated['Token'] = [tok.text for tok in doc]
text_annotated['Lemma'] = [tok.lemma_ for tok in doc]

# convert the dictionary to a dataframe 
text_annotated_df = pd.DataFrame(text_annotated)

# calculate how long the annotation and extraction took and print result
took = time() - current
print(f"Die Annotation hat {round(took, 2)} Sekunden gedauert.") 

Auszug aus der Tabelle, in der der annotierte Text gespeichert ist:

In [ ]:
# print first five lines of the annotation
text_annotated_df.head()

### Worthäufigkeit mit echter Tokenisierung   

Durch die Tokenisierung wurden z. B. Satzzeichen von Wörtern abgetrennt. An der Textlänge lässt sich dies schon erkennen. 

In [ ]:
# get the lemmata 
text_tokenized = text_annotated_df.Lemma

# print the length
len(text_tokenized)

Auf Grundlage des tokenisierten und lemmatisierten Textes, kann die Häufigkeitenabfrage erneut ausgeführt werden. Da durch die Lemmatisierung flektierte Wortformen auf die Grundformen zurückgeführt wurden, erwarten wir, dass die Häufigkeit einer Wortgrundform im Gegensatz zur vorherigen Abfrage erhöht ist. 

In [ ]:
# Count the words with Counter and save the result to a variable
token_frequencies = Counter(text_tokenized)

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Um die Häufigkeit nur mit Python abzufragen, kann folgende Zeile ausgeführt werden:
</details>

In [ ]:
# 🚀 get the number of the word "Grippe" in the word frequencies 
token_frequencies["Grippe"]

In [ ]:
# Ensure Bokeh output is displayed in the notebook
output_notebook()

# Convert the dictionary to a JSON string
tok_freq_json = json.dumps(token_frequencies)

# Create the text input widget
token_input = TextInput(value='', title="Geben Sie ein Wort ein:")

# Create a Div to display the frequency
token_frequency_display = Div(text="Häufigkeit: ")

# JavaScript callback to update the frequency display
# Only needed for graphical interface 
tok_callback = CustomJS(args=dict(frequency_display=token_frequency_display, text_input=token_input), code=f"""
    var tok = text_input.value.trim();

    // Parse the word frequency dictionary from Python
    var word_freq = {tok_freq_json};

    var frequency = tok in word_freq ? word_freq[tok] : "Nicht gefunden";
    frequency_display.text = "Häufigkeit: " + frequency;
""")

token_input.js_on_change('value', tok_callback)

# Layout and display
layout = column(token_input, token_frequency_display)
show(layout)

### 🚀 Selbst ausprobieren

This interactive element below demonstrates spaCy's lemmatization process using some selected examples from the newspaper text analysed in this chapter. Choose a token from the dropdown menu to see the corresponding lemma. Observe how inflected forms such as plurals and verb forms are reduced to their base form, while some tokens remain unchanged because they already correspond to their lemma.

In [ ]:
from IPython.display import display, HTML

display(HTML(r"""
<div id="lemma-demo" style="font-family: system-ui; max-width:700px;">

<select id="tokenSelect" style="padding:6px; margin-bottom:15px;">
  <option value="">Token auswählen...</option>
</select>

<div id="lemmaOutput"
     style="padding:12px; border:1px solid #ddd; border-radius:6px;">
</div>

</div>

<script>

(function() {

const data = {
  "Berlin":"Berlin",
  "Deutschen":"deutsch",
  "Franzosen":"Franzose",
  "Städten":"Stadt",
  "Kämpfen":"Kampf",
  "gefordert":"fordern",
  "verwüstet":"verwüsten",
  "eingetroffen":"eintreffen",
  "genannt":"nennen",
  "durchgeführt":"durchführen",
  "größere":"groß",
  "friedliche":"friedlich",
  "verantwortlich":"verantwortlich",
  "Regierung":"Regierung",
  "Bedingungen":"Bedingung"
};

const select = document.querySelector("#tokenSelect");

Object.keys(data).forEach(token => {
  let opt = document.createElement("option");
  opt.value = token;
  opt.textContent = token;
  select.appendChild(opt);
});

select.addEventListener("change", function() {

  const token = this.value;

  if(!token){
    document.getElementById("lemmaOutput").innerHTML = "";
    return;
  }

  const lemma = data[token];

  let note = "";

  if(token.toLowerCase() === lemma.toLowerCase()){
    note = `
      <p style="color:#c62828;font-weight:bold;">
      ℹ Token und Lemma sind identisch.
      </p>
    `;
  }

  document.getElementById("lemmaOutput").innerHTML = `
    <b>Ausgewähltes Token:</b> ${token}<br><br>
    <b>Lemma:</b> ${lemma}
    ${note}
  `;
});

})();

</script>
"""))

## Annotation speichern
Um den annotierten Text zu speichern, wird zuerst der Dateiname festgelegt. Dafür wird die Dateiendung ersetzt von `.txt` zu `.csv`.

[CSV](https://de.wikipedia.org/wiki/CSV_(Dateiformat)) (comma-separated value) ist das Standardformat um tabellarische Daten im Klartext zu speichern. 

<details>
  <summary><b>Informationen zum Ausführen des Notebooks</b></summary>
Um dieselbe Ordnerstruktur wie in dem GitHub-Repositorium zu erhalten wird in der nächsten Zelle ein Ordner `data` erstellt, in dem ein Ordner `csv` erstellt wird. In dem Ordner `csv` wird die Annotation gespeichert. 
</details>

In [ ]:
# 🚀 Create output folder 
output_dir = Path(r"../data/csv")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# set output path, change file extension
output_path = Path(r"../data/csv") / text_path.with_suffix(".csv").name

Der Text wird dann unter dem festgelegten Dateinamen gespeichert. 

In [ ]:
# save the annotation as csv
text_annotated_df.to_csv(output_path, index=False)

## Prozess für das gesamte Korpus ausführen 
```{admonition} Dauer der Annotation
:class: zeitinfo
Die Annotation des gesamten Korpus kann je nach Leistungsfähigkeit und Arbeitsspeicher (RAM) des Computers mehrere Stunden in Anspruch nehmen.
```

In [ ]:
def stream_texts_from_directory(corpus_filepaths: list[Path]) -> typing.Generator[str, None, None]:
    """A generator that yields texts from files in the file list one by one."""
    for filepath in corpus_filepaths:
            yield filepath.read_text(encoding="utf-8")

def process_corpus(corpus_dir: Path, output_dir: Path, batch_size=15) -> None:
    """
    Reads files from corpus_dir, annotates the files with spacy and writes the result
    to the output_dir
    :param Path corpus_dir: The directory in which the txt files are saved
    :param Path output_dir: The directory in which the annotations are written to as csv
    """
    corpus_filepaths = [f for f in corpus_dir.iterdir() if f.is_file() and f.suffix == ".txt"]

    start = time()
    for filepath, doc in zip(corpus_filepaths, nlp.pipe(stream_texts_from_directory(corpus_filepaths), 
                                                        disable=disable_components, batch_size=batch_size)):
        print(filepath)
        # Save the token and lemma information to a dictionary
        text_annotated = {}
        text_annotated['Token'] = [tok.text for tok in doc]
        text_annotated['Lemma'] = [tok.lemma_ for tok in doc]
        annotation_df = pd.DataFrame(text_annotated)
        
        output_path = output_dir / filepath.with_suffix(".csv").name
        annotation_df.to_csv(output_path, index=False)
    end = time()
    
    print(f"""Processed {len(corpus_filepaths)} texts with spacy.
    Took {round((end - start)  / 60, 4)} minutes in total.""")

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Im folgenden werden alle Textdateien im Korpus heruntergeladen und gespeichert. Dafür sind folgende Schritte nötig:
<ol>
    <li>Es wird eine Liste erstellt, die die URLs zu den einzelnen Textdateien beinhaltet.</li>
    <li>Alle Dateien aus der Liste werden heruntergeladen und in dem Ordner <i>../data/txt</i> gespeichert.</li>
</ol>
Sollten die Dateien schon an einem anderen Ort vorhanden sein, können die Dateipfade zu den Ordnern angepasst werden. </br>
Des Weiteren wird der Ordner für die annotierten Dateien angelegt: <i>../data/csv</i>
</details>

In [ ]:
# 🚀 Create download list 
github_api_txt_dir_path = "https://api.github.com/repos/quadriga-dk/Text-Fallstudie-1/contents/data/txt"
txt_dir_info = requests.get(github_api_txt_dir_path).json()
url_list = [entry["download_url"] for entry in txt_dir_info]

In [ ]:
# ⚠️ Only execute, if you haven't downloaded the files yet!
# 🚀 Download all txt files – this step will take a while
for url in tqdm(url_list, desc="Downloading txt files"):
    download_file(url, "../data/txt")

In [ ]:
# 🚀 Create output folder 
# if create before, this cell can be skipped
output_dir = Path(r"../data/csv")
output_dir.mkdir(parents=True, exist_ok=True)

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Bei Computern mit kleinerem Arbeitsspeicher (weniger als 12 GB), sollte der Parameter „batch_size“ verkleinert werden (z.B. auf 5), da der Arbeitsspeicher ansonsten 'überlaufen' könnte (das nennt sich auch „out of memory“). Das bedeutet, es gibt nicht mehr genug Arbeitsspeicher, um die Annotation wie auch andere laufende Programme durchzuführen. Der Computer wird dann langsamer oder reagiert nicht mehr.
</details>

In [ ]:
# Set path to corpus and output dir
corpus_dir = Path(r"../data/txt/")
output_dir = Path(r"../data/csv")

# Read, annotate, write 
process_corpus(corpus_dir, output_dir, batch_size=15)